# 🌐 Assistente Diversa — Educação Inclusiva
## Protótipo funcional para TCC

Este notebook entrega o protótipo funcional do TCC: um chat de Educação Inclusiva que usa uma base de conhecimento do Portal Diversa, busca por relevância com TF-IDF e geração de resposta com Llama 3.1 via Groq.

**Importante:** se você não informar a chave da Groq, o protótipo ainda funciona em modo demonstração, respondendo com base nos trechos recuperados da base.

## 1. Instalação de dependências



In [1]:
!pip install -q beautifulsoup4 gradio python-dotenv

print("✅ Instalação concluída!")
print()
print("Bibliotecas que já vêm no Colab:")
print("  • requests   — chamadas HTTP (coleta e API)")
print("  • pandas     — manipulação de dados")
print("  • re         — expressões regulares (limpeza de texto)")
print("  • sklearn    — TfidfVectorizer e cosine_similarity")


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Reshimming asdf python...
✅ Instalação concluída!

Bibliotecas que já vêm no Colab:
  • requests   — chamadas HTTP (coleta e API)
  • pandas     — manipulação de dados
  • re         — expressões regulares (limpeza de texto)
  • sklearn    — TfidfVectorizer e cosine_similarity


## 2. Importações e configuração



In [2]:
import json
import os
import re
import time
from typing import Dict, List

import gradio as gr
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv

load_dotenv()
GROQ_KEY = os.getenv("GROQ_KEY", "")

GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

PARAMS_GERACAO = {
    "model": GROQ_MODEL,
    "max_tokens": 600,
    "temperature": 0.3,
    "top_p": 0.9,
}

print("✅ Configuração carregada")
print("Modelo:", GROQ_MODEL)
print("Modo:", "Groq/Llama" if GROQ_KEY else "Demonstração sem API")

/home/joaotrovaum/.asdf/installs/python/3.11.9/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


✅ Configuração carregada
Modelo: llama-3.1-8b-instant
Modo: Groq/Llama


## 3. Base de conhecimento


In [ ]:
artigos_brutos = [
    # ============================================================
    # AUTISMO / TEA
    # ============================================================
    {
        "titulo": "TEA na educação infantil: inclusão e afetividade na prática docente",
        "url": "https://diversa.org.br/artigos/tea-educacao-infantil/",
        "texto": "O artigo aborda a inclusão de crianças com Transtorno do Espectro Autista na educação infantil, destacando a importância da qualificação docente, da afetividade e da criação de vínculos no processo de aprendizagem. A proposta reforça que a educação inclusiva deve considerar as singularidades de cada criança, respeitando seus modos de comunicação, interação e participação. O texto também mostra que estratégias pensadas para estudantes com TEA podem beneficiar toda a turma quando são planejadas de forma acolhedora e flexível."
    },
    {
       "titulo": "Ferramentas para o desenvolvimento escolar de estudantes com TEA",
       "url": "https://diversa.org.br/artigos/ferramentas-para-o-desenvolvimento-escolar-de-estudantes-com-tea/",
       "texto": "O artigo apresenta estratégias que podem apoiar o desenvolvimento escolar de estudantes com Transtorno do Espectro Autista. O texto destaca que o TEA envolve diferentes formas de interação social, comunicação e comportamento, exigindo que a escola observe o estudante de maneira individualizada. Também reforça a importância de recursos pedagógicos, planejamento, rotina, acolhimento e práticas que favoreçam a participação do estudante com TEA na sala comum, evitando sua exclusão das atividades escolares."
    },

    # ============================================================
    # TDAH
    # ============================================================
    {
        "titulo": "O que é Transtorno do Déficit de Atenção e Hiperatividade (TDAH)?",
        "url": "https://diversa.org.br/artigos/o-que-e-transtorno-do-deficit-de-atencao-e-hiperatividade-tdah/",
        "texto": "O artigo explica o Transtorno do Déficit de Atenção e Hiperatividade, conhecido como TDAH, destacando características como desatenção, hiperatividade e impulsividade. No contexto escolar, o texto aponta que essas características podem interferir na concentração, organização e realização das atividades. A abordagem inclusiva exige que a escola não reduza o estudante ao diagnóstico, mas busque estratégias pedagógicas, combinados claros, flexibilização de propostas e formas de apoiar sua participação e aprendizagem."
    },
    {
        "titulo": "Como a escola pode lidar com os diagnósticos mentais na infância",
        "url": "https://diversa.org.br/artigos/como-a-escola-pode-lidar-com-os-diagnosticos-mentais-na-infancia/",
        "texto": "O artigo propõe uma reflexão cuidadosa sobre diagnósticos mentais na infância, incluindo discussões relacionadas ao TDAH e aos chamados distúrbios de atenção. A autora alerta para a necessidade de cautela diante de rótulos e classificações que podem limitar a forma como a escola enxerga a criança. Para a educação inclusiva, é importante compreender o estudante em sua totalidade, observar o contexto, respeitar o tempo da infância e construir práticas pedagógicas que não reduzam a criança ao diagnóstico."
    },

    # ============================================================
    # DEFICIÊNCIA VISUAL
    # ============================================================
    {
        "titulo": "Psicomotricidade como recurso para a educação na deficiência visual",
        "url": "https://diversa.org.br/artigos/psicomotricidade-como-recurso-para-a-educacao-na-deficiencia-visual/",
        "texto": "O artigo apresenta a psicomotricidade como um recurso importante para a educação de estudantes com deficiência visual. A proposta considera o desenvolvimento motor, cognitivo, afetivo e sensorial, valorizando diferentes canais de aprendizagem, como tato, audição, movimento, equilíbrio e percepção corporal. Na perspectiva inclusiva, o texto reforça que a escola deve adaptar práticas, espaços e estratégias para garantir que estudantes com deficiência visual participem das atividades com autonomia, segurança e protagonismo."
    },
    {
        "titulo": "Braile: acessibilidade para estudantes com deficiência visual",
        "url": "https://diversa.org.br/noticias/braile-acessibilidade-para-estudantes-com-deficiencia-visual/",
        "texto": "O conteúdo aborda o uso do braile como recurso de acessibilidade para estudantes com deficiência visual, mostrando experiências em que escolas desenvolveram materiais pedagógicos acessíveis para favorecer a participação plena dos alunos. O texto destaca que o braile, os materiais táteis, a audiodescrição, a colaboração entre professores e o Atendimento Educacional Especializado podem ampliar o acesso ao currículo. A inclusão acontece quando os recursos são planejados para envolver toda a turma, e não apenas um estudante isoladamente."
    },

    # ============================================================
    # DEFICIÊNCIA AUDITIVA
    # ============================================================
    {
        "titulo": "Educação de surdos: os desafios da proposta bilíngue",
        "url": "https://diversa.org.br/artigos/educacao-de-surdos-os-desafios-da-proposta-bilingue/",
        "texto": "O artigo discute a educação de estudantes surdos e os desafios da proposta bilíngue, reconhecendo a importância da Língua Brasileira de Sinais no processo educacional. O texto destaca que a Libras é fundamental para comunicação, aprendizagem e participação social de muitos estudantes surdos. A escola inclusiva precisa considerar a língua de sinais, a língua portuguesa escrita, a formação dos profissionais, os materiais acessíveis e a valorização da cultura surda para garantir acesso real ao currículo."
    },
    {
        "titulo": "Qual o papel do Tradutor/Intérprete de Libras na escola inclusiva?",
        "url": "https://diversa.org.br/artigos/qual-o-papel-do-tradutor-interprete-de-libras-na-escola-inclusiva/",
        "texto": "O artigo explica a importância do tradutor e intérprete de Libras na inclusão de pessoas com deficiência auditiva, especialmente em contextos educacionais. Esse profissional atua na mediação linguística e cultural entre Libras e Língua Portuguesa, favorecendo a comunicação e a participação dos estudantes surdos. No entanto, o texto também permite compreender que a inclusão não depende apenas do intérprete: toda a escola precisa se responsabilizar pela acessibilidade comunicacional, pelo planejamento pedagógico e pela participação do estudante."
    },

    # ============================================================
    # DEFICIÊNCIA INTELECTUAL
    # ============================================================
    {
        "titulo": "Flexibilizações vs. adaptações curriculares: como incluir alunos com deficiência intelectual",
        "url": "https://diversa.org.br/artigos/flexibilizacoes-adaptacoes-curriculares-como-incluir-alunos-deficiencia-intelectual/",
        "texto": "O artigo discute a inclusão de estudantes com deficiência intelectual a partir da diferença entre flexibilizações e adaptações curriculares. A proposta inclusiva defende que o estudante tenha acesso ao currículo comum, com diferentes estratégias, tempos, recursos e formas de participação. O texto alerta que reduzir conteúdos de forma automática pode reforçar exclusões. Em vez disso, a escola deve conhecer o aluno, identificar barreiras e diversificar as práticas pedagógicas para garantir aprendizagem e participação."
    },
    {
        "titulo": "Os desafios de ser uma mulher com deficiência intelectual",
        "url": "https://diversa.org.br/artigos/os-desafios-de-ser-uma-mulher-com-deficiencia-intelectual/",
        "texto": "O artigo traz o relato de uma mulher com deficiência intelectual que enfrentou barreiras, discriminação e capacitismo ao longo de sua trajetória escolar e profissional. A narrativa destaca a importância do apoio da família, de educadores e de pessoas que acreditaram em seu potencial. Para a educação inclusiva, o texto contribui ao mostrar que estudantes com deficiência intelectual não devem ser definidos por limitações, mas reconhecidos como sujeitos de direitos, capazes de aprender, participar, desenvolver talentos e construir autonomia."
    },

    # ============================================================
    # DEFICIÊNCIA FÍSICA
    # ============================================================
    {
        "titulo": "Dicas e práticas para inclusão de alunos com deficiência física",
        "url": "https://diversa.org.br/noticias/dicas-praticas-inclusao-alunos-deficiencia-fisica/",
        "texto": "O conteúdo apresenta práticas e reflexões sobre a inclusão de estudantes com deficiência física na escola regular. A ideia central é que todo corpo pode participar das atividades escolares quando a escola elimina barreiras e adapta estratégias. O texto reúne experiências de educadores que buscaram garantir participação, movimento, aprendizagem e convivência, mostrando que a deficiência física não deve ser motivo para separar o estudante, mas para repensar espaços, materiais, atitudes e formas de participação."
    },
    {
        "titulo": "Educação física como um meio para a inclusão social e qualidade de vida",
        "url": "https://diversa.org.br/artigos/educacao-fisica-como-um-meio-para-a-inclusao-social-e-qualidade-de-vida/",
        "texto": "O artigo aborda a educação física escolar na perspectiva inclusiva, defendendo o direito de todos os estudantes, com ou sem deficiência, participarem das aulas. No caso de estudantes com deficiência física, a proposta exige reconhecer suas possibilidades, adaptar práticas quando necessário e valorizar a participação coletiva. O texto reforça que a educação física não deve ser confundida apenas com rendimento esportivo, mas entendida como componente curricular que contribui para convivência, qualidade de vida, respeito às diferenças e inclusão social."
    },

    # ============================================================
    # Textos Institucionais
    # ============================================================
    {
        "titulo": "Sobre o Diversa — O que é o projeto",
        "url": "https://diversa.org.br/sobre-o-projeto/",
        "texto": "O DIVERSA é uma plataforma digital voltada à construção e ao compartilhamento de conhecimentos sobre boas práticas de educação inclusiva. Seu público-alvo inclui educadores, gestores escolares, técnicos de secretarias de educação, familiares de estudantes e outros profissionais comprometidos com o tema. A plataforma reúne relatos de experiência, estudos de caso, pesquisas, artigos, notícias e materiais que podem apoiar redes de ensino interessadas em oferecer uma educação de qualidade para todos os estudantes, respeitando suas especificidades."
    },
    {
        "titulo": "O que é educação inclusiva?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-educacao-inclusiva",
        "texto": "A educação inclusiva pode ser compreendida como um conjunto de processos políticos, pedagógicos e sociais que buscam garantir o direito de todos os estudantes à aprendizagem. Essa perspectiva exige o compromisso da comunidade escolar e a transformação de práticas tradicionais que tratam os alunos como se todos aprendessem da mesma forma. A proposta inclusiva considera as diversidades intelectuais, físicas, sensoriais, sociais, raciais e de gênero, buscando eliminar barreiras e ampliar a participação de todos."
    },
    {
        "titulo": "Quais os princípios da educação inclusiva?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-educacao-inclusiva/quais-os-principios-da-educacao-inclusiva/",
        "texto": "Os princípios da educação inclusiva ajudam escolas, famílias e profissionais da educação a avaliarem se suas práticas realmente garantem o direito à educação para todos. Entre os princípios apresentados pelo Instituto Rodrigo Mendes estão a ideia de que toda pessoa tem direito à educação de qualidade, toda pessoa aprende, cada processo de aprendizagem é singular, o convívio no ambiente comum beneficia todos e a educação inclusiva diz respeito a toda a comunidade escolar, não apenas aos estudantes com deficiência."
    },
    {
        "titulo": "Qual é o histórico da educação inclusiva?",
        "url": "https://diversa.org.br/educacao-inclusiva/qual-e-o-historico-da-educacao-inclusiva/",
        "texto": "A história da educação inclusiva envolve diferentes formas de compreender a deficiência e o direito à participação social. O Portal Diversa apresenta paradigmas como exclusão, segregação, integração e inclusão, mostrando que a perspectiva inclusiva representa uma mudança importante: a escola deixa de exigir que o estudante se adapte sozinho a um padrão e passa a assumir a responsabilidade de eliminar barreiras, oferecer oportunidades e reconhecer a educação como um direito humano fundamental."
    },
    {
        "titulo": "Quem são as pessoas com deficiência?",
        "url": "https://diversa.org.br/educacao-inclusiva/quem-sao-as-pessoas-com-deficiencia/",
        "texto": "O conceito de pessoa com deficiência mudou ao longo do tempo. Na perspectiva inclusiva, a deficiência não é entendida apenas como uma característica individual, mas como resultado da interação entre impedimentos de longo prazo e barreiras existentes no ambiente. Essas barreiras podem ser arquitetônicas, comunicacionais, pedagógicas, tecnológicas ou atitudinais. Assim, a inclusão exige que a sociedade e a escola modifiquem suas estruturas para garantir participação plena e igualdade de oportunidades."
    },
    {
        "titulo": "O que muda na educação especial com a perspectiva da inclusão?",
        "url": "https://diversa.org.br/educacao-inclusiva/qual-e-o-historico-da-educacao-inclusiva/o-que-muda-na-educacao-especial-com-a-perspectiva-da-inclusao/",
        "texto": "Com a perspectiva da inclusão, a educação especial deixa de funcionar como um sistema separado ou substitutivo da escola comum. Ela passa a ter caráter complementar, suplementar e transversal, apoiando o ensino regular na eliminação de barreiras à participação dos estudantes. Essa mudança reforça que estudantes com deficiência, Transtorno do Espectro Autista e altas habilidades ou superdotação devem estar matriculados na escola comum, com acesso a recursos, serviços e estratégias que favoreçam sua aprendizagem."
    },
    {
        "titulo": "O que é Atendimento Educacional Especializado (AEE)?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-o-atendimento-educacional-especializado-aee/",
        "texto": "O Atendimento Educacional Especializado, conhecido como AEE, é um serviço da educação especial que identifica, elabora e organiza recursos pedagógicos e de acessibilidade para eliminar barreiras à participação dos estudantes. Ele deve complementar ou suplementar a escolarização, nunca substituir a sala de aula comum. O AEE deve ocorrer de forma articulada com professores, equipe pedagógica e famílias, contribuindo para a autonomia e a aprendizagem dos estudantes público-alvo da educação especial."
    },
    {
        "titulo": "Como o AEE contribui para um planejamento pedagógico inclusivo?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-o-atendimento-educacional-especializado-aee/como-o-atendimento-educacional-especializado-pode-contribuir-para-um-planejamento-pedagogico-inclusivo/",
        "texto": "O Atendimento Educacional Especializado pode contribuir para o planejamento pedagógico inclusivo ao ajudar na identificação de barreiras e na escolha de recursos, estratégias e formas de participação. Embora atenda diretamente estudantes público-alvo da educação especial, o trabalho do profissional de AEE pode beneficiar toda a turma, pois as estratégias pensadas para um estudante muitas vezes ampliam as possibilidades de aprendizagem de outros alunos."
    },
    {
        "titulo": "O que são as Salas de Recursos Multifuncionais (SRM)?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-o-atendimento-educacional-especializado-aee/o-que-sao-as-salas-de-recursos-multifuncionais-srm/",
        "texto": "As Salas de Recursos Multifuncionais são espaços existentes em escolas de educação básica onde ocorre, preferencialmente, o Atendimento Educacional Especializado. Esses espaços contam com equipamentos, materiais pedagógicos e recursos de acessibilidade que ajudam a eliminar barreiras e a potencializar o processo de escolarização dos estudantes. A função da sala de recursos não é separar o estudante, mas oferecer apoio para sua participação na sala comum e nos demais espaços escolares."
    },
    {
        "titulo": "O que são Tecnologias Assistivas (TAs)?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-sao-tecnologias-assistivas-tas/",
        "texto": "Tecnologias Assistivas são produtos, recursos, serviços, técnicas, estratégias e práticas que ajudam a ampliar habilidades funcionais de pessoas com deficiência. No contexto escolar, elas podem contribuir para mobilidade, comunicação, acesso à informação, participação e autonomia. Podem ser recursos simples, como adaptações em materiais escolares, ou tecnologias mais complexas, como leitores de tela, softwares, equipamentos de comunicação e outros instrumentos de acessibilidade."
    },
    {
        "titulo": "O que é a Lei Brasileira de Inclusão (LBI)?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-a-lei-brasileira-de-inclusao-lbi/",
        "texto": "A Lei Brasileira de Inclusão, também conhecida como Estatuto da Pessoa com Deficiência, busca assegurar e promover, em igualdade de condições, o exercício dos direitos e das liberdades fundamentais da pessoa com deficiência. Na área da educação, a LBI reforça o direito à matrícula, à participação e à permanência em escolas regulares, tanto públicas quanto privadas, além de prever acessibilidade, profissional de apoio quando necessário e combate à discriminação."
    },
    {
        "titulo": "O que é avaliação na perspectiva inclusiva?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-avaliacao-na-perspectiva-inclusiva/",
        "texto": "A avaliação na perspectiva inclusiva parte da ideia de que todos os estudantes aprendem, mas cada pessoa possui um processo singular de aprendizagem. Quando um aluno apresenta dificuldades, a escola deve investigar quais barreiras podem estar impedindo sua participação e seu acesso ao conhecimento. Essa visão desloca o foco da limitação do estudante para a responsabilidade da escola em diversificar estratégias, flexibilizar práticas e buscar formas mais justas de acompanhar o desenvolvimento de cada aluno."
    },
    # ============================================================
    # LBI - LEI BRASILEIRA DE INCLUSÃO
    # ============================================================
    {
        "titulo": "O que é a Lei Brasileira de Inclusão (LBI)?",
        "url": "https://diversa.org.br/educacao-inclusiva/o-que-e-a-lei-brasileira-de-inclusao-lbi/",
        "texto": "A Lei Brasileira de Inclusão, também conhecida como Estatuto da Pessoa com Deficiência, é um marco legal brasileiro que busca assegurar o exercício dos direitos e das liberdades fundamentais das pessoas com deficiência em igualdade de condições. No campo educacional, a LBI reforça o direito à matrícula, à permanência, à participação e à aprendizagem em escolas comuns. A lei também combate práticas discriminatórias, como a recusa de matrícula, a cobrança de valores adicionais e a falta de recursos de acessibilidade necessários para garantir a inclusão escolar."
    },
    {
        "titulo": "Lei Brasileira de Inclusão (LBI) e o acesso à educação",
        "url": "https://diversa.org.br/artigos/lei-brasileira-de-inclusao-lbi-e-o-acesso-a-educacao/",
        "texto": "O artigo apresenta a relação entre a Lei Brasileira de Inclusão e o direito à educação das pessoas com deficiência. A LBI estabelece que sistemas de ensino devem garantir condições de acesso, permanência, participação e aprendizagem, com oferta de recursos de acessibilidade e apoios necessários. O texto destaca que a inclusão não se resume à matrícula, pois exige ações concretas da escola para eliminar barreiras, promover convivência, adaptar práticas pedagógicas e assegurar que estudantes com deficiência tenham acesso ao currículo e à vida escolar."
    },

    # ============================================================
    # LDB - LEI DE DIRETRIZES E BASES DA EDUCAÇÃO NACIONAL
    # ============================================================
    {
        "titulo": "A legislação federal brasileira e a educação de alunos com deficiência",
        "url": "https://diversa.org.br/artigos/a-legislacao-federal-brasileira-e-a-educacao-de-alunos-com-deficiencia/",
        "texto": "O artigo apresenta marcos legais da educação de estudantes com deficiência no Brasil, incluindo a Lei de Diretrizes e Bases da Educação Nacional, Lei nº 9.394 de 1996. A LDB define a educação especial como modalidade de ensino e prevê atendimento aos educandos com deficiência, transtornos globais do desenvolvimento e altas habilidades ou superdotação. A partir da perspectiva inclusiva, a educação especial deve atuar de forma articulada à escola comum, oferecendo apoios, recursos e serviços que favoreçam a participação e a aprendizagem."
    },
    {
        "titulo": "Educação inclusiva e formação docente",
        "url": "https://diversa.org.br/artigos/educacao-inclusiva-e-formacao-docente/",
        "texto": "O artigo discute a formação docente para a educação inclusiva e relaciona esse tema à Lei de Diretrizes e Bases da Educação Nacional. A LDB aponta que os sistemas de ensino devem assegurar professores com formação adequada para o atendimento especializado e professores do ensino regular capacitados para atuar com estudantes público-alvo da educação especial nas classes comuns. O texto reforça que a inclusão depende de uma escola democrática, capaz de respeitar diferentes tempos, trajetórias e formas de aprendizagem."
    },

    # ============================================================
    # BNCC - BASE NACIONAL COMUM CURRICULAR
    # ============================================================
    {
        "titulo": "O que é a Base Nacional Comum Curricular (BNCC)?",
        "url": "https://diversa.org.br/educacao-inclusiva/qual-a-importancia-do-curriculo-na-educacao-inclusiva/o-que-a-a-base-nacional-comum-curricular-bncc/",
        "texto": "A Base Nacional Comum Curricular define as aprendizagens essenciais às quais todos os estudantes têm direito ao longo da educação básica. Na perspectiva inclusiva, a BNCC deve ser compreendida como referência para garantir acesso ao currículo comum, sem excluir estudantes com deficiência, transtorno do espectro autista, altas habilidades ou outras especificidades. Isso significa que a escola deve organizar estratégias, recursos e formas de participação para que todos possam aprender a partir de suas possibilidades."
    },
    {
        "titulo": "Como a Base Nacional Comum Curricular (BNCC) contribui para a educação inclusiva?",
        "url": "https://diversa.org.br/educacao-inclusiva/qual-a-importancia-do-curriculo-na-educacao-inclusiva/como-a-base-nacional-comum-curricular-bncc-contribui-para-a-educacao-inclusiva/",
        "texto": "O conteúdo explica que a BNCC contribui para a educação inclusiva ao afirmar que todos os estudantes têm direito às aprendizagens essenciais. A inclusão não significa oferecer um currículo inferior ou separado para estudantes com deficiência, mas garantir acesso ao mesmo currículo por diferentes caminhos. Para isso, a escola precisa diversificar estratégias, flexibilizar práticas, produzir materiais acessíveis e reconhecer que cada estudante pode demonstrar aprendizagem de maneiras distintas."
    },

    # ============================================================
    # DECRETO 12.686/2025 - PNEEI
    # ============================================================
    {
        "titulo": "Decreto nº 12.686/2025 — Política Nacional de Educação Especial Inclusiva",
        "url": "https://www.planalto.gov.br/ccivil_03/_ato2023-2026/2025/decreto/d12686.htm",
        "texto": "O Decreto nº 12.686, de 20 de outubro de 2025, institui a Política Nacional de Educação Especial Inclusiva e a Rede Nacional de Educação Especial Inclusiva. A política tem a finalidade de garantir o direito à educação em um sistema educacional inclusivo para estudantes com deficiência, com transtorno do espectro autista e com altas habilidades ou superdotação. O decreto reforça que a educação especial deve ser oferecida de forma transversal, com recursos e serviços que apoiem, complementem e suplementem o processo de escolarização em classes e escolas comuns."
    },
    {
        "titulo": "Governo institui a Política Nacional de Educação Especial Inclusiva",
        "url": "https://www.gov.br/mec/pt-br/assuntos/noticias/2025/outubro/governo-institui-a-politica-nacional-de-educacao-especial-inclusiva",
        "texto": "A notícia do Ministério da Educação apresenta a instituição da Política Nacional de Educação Especial Inclusiva, criada pelo Decreto nº 12.686/2025. Segundo o MEC, a nova política busca fortalecer a inclusão escolar e organizar ações voltadas ao acesso, à permanência, à participação e à aprendizagem de estudantes público-alvo da educação especial. O texto destaca que a política será coordenada pelo MEC e envolve a criação da Rede Nacional de Educação Especial Inclusiva para apoiar redes de ensino na implementação de práticas inclusivas."
    },

    # ============================================================
    # PNEEPEI - POLÍTICA DE 2008
    # ============================================================
    {
        "titulo": "O que é a Política Nacional de Educação Especial na Perspectiva da Educação Inclusiva?",
        "url": "https://diversa.org.br/educacao-inclusiva/qual-e-a-legislacao-atual-sobre-educacao-inclusiva-no-brasil/o-que-e-a-politica-nacional-de-educacao-especial-na-perspectiva-da-educacao-inclusiva/",
        "texto": "A Política Nacional de Educação Especial na Perspectiva da Educação Inclusiva, criada em 2008, é um marco para a garantia da matrícula de estudantes com deficiência na escola comum. A política orienta que a educação especial integre a proposta pedagógica da escola, eliminando barreiras para a participação de estudantes com deficiência, transtorno do espectro autista e altas habilidades ou superdotação. Também define o Atendimento Educacional Especializado como serviço de apoio à escolarização, sem substituir a sala comum."
    },
    {
        "titulo": "Por uma educação na perspectiva inclusiva",
        "url": "https://diversa.org.br/artigos/por-uma-educacao-na-perspectiva-inclusiva/",
        "texto": "O artigo analisa a importância da Política Nacional de Educação Especial na Perspectiva da Educação Inclusiva, conhecida como PNEEPEI, destacando sua relevância para ampliar matrículas de estudantes público-alvo da educação especial em classes comuns. O texto defende que a educação deve incorporar a perspectiva inclusiva de forma ampla, democrática e acolhedora da diversidade. Também reforça que a PNEEPEI ajudou a transformar a educação especial, deslocando-a de um modelo separado para uma atuação articulada à escola comum."
    }
]


print(f"✅ Base carregada com {len(artigos_brutos)} textos")
for i, artigo in enumerate(artigos_brutos, 1):
     print(f"{i:02d}. {artigo['titulo']}")

✅ Base carregada com 34 textos
01. TEA na educação infantil: inclusão e afetividade na prática docente
02. Ferramentas para o desenvolvimento escolar de estudantes com TEA
03. O que é Transtorno do Déficit de Atenção e Hiperatividade (TDAH)?
04. Como a escola pode lidar com os diagnósticos mentais na infância
05. Psicomotricidade como recurso para a educação na deficiência visual
06. Braile: acessibilidade para estudantes com deficiência visual
07. Educação de surdos: os desafios da proposta bilíngue
08. Qual o papel do Tradutor/Intérprete de Libras na escola inclusiva?
09. Flexibilizações vs. adaptações curriculares: como incluir alunos com deficiência intelectual
10. Os desafios de ser uma mulher com deficiência intelectual
11. Dicas e práticas para inclusão de alunos com deficiência física
12. Educação física como um meio para a inclusão social e qualidade de vida
13. Sobre o Diversa — O que é o projeto
14. O que é educação inclusiva?
15. Quais os princípios da educação inclusiva?
1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Limpeza da base



In [ ]:
def limpar_texto(texto):
    texto = re.sub(r"\s+", " ", str(texto))
    texto = re.sub(r"\[.*?\]", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()

df = pd.DataFrame(artigos_brutos)
df["texto_limpo"] = df["texto"].apply(limpar_texto)
df = df[df["texto_limpo"].str.len() > 120].reset_index(drop=True)
df["conteudo_busca"] = df["titulo"] + ". " + df["titulo"] + ". " + df["texto_limpo"]

print("✅ Limpeza concluída")
print("Total de artigos válidos:", len(df))
display(df[["titulo", "url"]])

✅ Limpeza concluída
Total de artigos válidos: 34


,titulo,url
0,TEA na educação infantil: inclusão e afetivida...,https://diversa.org.br/artigos/tea-educacao-in...
1,Ferramentas para o desenvolvimento escolar de ...,https://diversa.org.br/artigos/ferramentas-par...
2,O que é Transtorno do Déficit de Atenção e Hip...,https://diversa.org.br/artigos/o-que-e-transto...
3,Como a escola pode lidar com os diagnósticos m...,https://diversa.org.br/artigos/como-a-escola-p...
4,Psicomotricidade como recurso para a educação ...,https://diversa.org.br/artigos/psicomotricidad...
5,Braile: acessibilidade para estudantes com def...,https://diversa.org.br/noticias/braile-acessib...
6,Educação de surdos: os desafios da proposta bi...,https://diversa.org.br/artigos/educacao-de-sur...
7,Qual o papel do Tradutor/Intérprete de Libras ...,https://diversa.org.br/artigos/qual-o-papel-do...
8,Flexibilizações vs. adaptações curriculares: c...,https://diversa.org.br/artigos/flexibilizacoes...
9,Os desafios de ser uma mulher com deficiência ...,https://diversa.org.br/artigos/os-desafios-de-...


## 5. Vetorização TF-IDF e busca semântica


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=True,
)

matriz_tfidf = vectorizer.fit_transform(df["conteudo_busca"])

def buscar_artigos_relevantes(pergunta, top_k=3, score_minimo=0.015):
    vetor = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor, matriz_tfidf)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultados = []
    for i in indices:
        score = float(scores[i])
        if score >= score_minimo:
            resultados.append({
                "titulo": df.iloc[i]["titulo"],
                "url": df.iloc[i]["url"],
                "trecho": df.iloc[i]["texto_limpo"][:900],
                "score": round(score, 4),
            })
    return resultados

print("✅ Vetorização concluída")
print("Matriz:", matriz_tfidf.shape)

✅ Vetorização concluída
Matriz: (34, 2417)


## 6. Teste da busca

In [ ]:
pergunta_teste = "O que é Atendimento Educacional Especializado?"
resultados = buscar_artigos_relevantes(pergunta_teste)

print("Pergunta:", pergunta_teste)
print("Artigos recuperados:")
for r in resultados:
        print(f"- {r['titulo']} | score={r['score']}")

Pergunta: O que é Atendimento Educacional Especializado?
Artigos recuperados:
- O que é Atendimento Educacional Especializado (AEE)? | score=0.3656
- O que é a Política Nacional de Educação Especial na Perspectiva da Educação Inclusiva? | score=0.1317
- Braile: acessibilidade para estudantes com deficiência visual | score=0.124


## 7. Prompt do assistente e controle de escopo


In [ ]:
SYSTEM_PROMPT = (
"Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br)."
"Seu papel é apoiar professores, gestores escolares, familiares de pessoas com deficiência, estudantes, "
        "pesquisadores e profissionais de secretarias de educação, usando os TRECHOS fornecidos."
        "RESPONDA apenas temas de educação inclusiva: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual "
        "e física, AEE, sala de recursos multifuncionais, tecnologia assistiva, acessibilidade, adaptações curriculares, "
        "DUA, legislação educacional, família, gestão escolar e conteúdos do Portal Diversa."
        "NÃO RESPONDA: diagnóstico médico ou psicológico, recomendação de medicamento/tratamento, opinião política "
        "partidária, decisão judicial individual, nem assuntos sem relação com educação inclusiva."
        "Nunca invente URLs, títulos, leis, números ou fontes. Cite apenas fontes presentes nos TRECHOS fornecidos. "
        "Se a base não tiver informação suficiente, diga isso claramente."
        "Responda sempre em português brasileiro, com linguagem clara, acessível, empática e prática."
    )

PALAVRAS_ESCOPO = [
        "educação", "inclus", "deficiência", "deficiente", "autismo", "tea", "tdah", "aee", "lbi", "bncc",
        "ldb", "aluno", "estudante", "escola", "professor", "gestor", "família", "sala de recursos",
        "tecnologia assistiva", "acessibilidade", "libras", "braille", "curricular", "pedagóg", "aprendizagem",
        "apoio escolar", "laudo", "diagnóstico", "barreira", "adaptação", "desenho universal", "dua"
    ]

def pergunta_dentro_escopo(pergunta):
        pergunta_normalizada = limpar_texto(pergunta).lower()
        return any(palavra in pergunta_normalizada for palavra in PALAVRAS_ESCOPO)

print("✅ Prompt configurado")

✅ Prompt configurado


## 8. Funções de resposta



In [ ]:
def chamar_groq(messages, tentativas=3):
    if not GROQ_KEY:
        return ""

    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type": "application/json",
    }
    payload = {**PARAMS_GERACAO, "messages": messages}

    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=30)
            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"].strip()
            if resp.status_code == 429:
                time.sleep(5 * tentativa)
                continue
            if resp.status_code == 401:
                return "❌ Chave da Groq inválida. Verifique a variável GROQ_KEY."
            return f"❌ Erro HTTP {resp.status_code}: {resp.text[:250]}"
        except Exception as exc:
            return f"❌ Erro inesperado: {exc}"
    return "⏳ Serviço indisponível no momento."

def resposta_demo_sem_llm(pergunta, artigos):
    if not artigos:
        return (
            "Minha especialidade é Educação Inclusiva. A base atual ainda não possui trechos suficientes "
            "para responder essa pergunta com segurança. Recomendo ampliar a base com mais artigos do Portal Diversa."
        )
    principal = artigos[0]
    return (
        "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
        f"Com base no artigo **{principal['titulo']}**, a orientação principal é considerar a educação inclusiva "
        "como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. "
        "A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado "
        "quando necessário e envolver professores, gestão, família e comunidade.\n\n"
        "Trecho recuperado da base:\n"
        f"> {principal['trecho'][:550]}...\n\n"
        "Para respostas mais completas geradas por IA, configure a variável `GROQ_KEY`."
    )

def montar_fontes(artigos):
    if not artigos:
        return ""
    linhas = ["\n\n---", "📚 **Fontes recuperadas da base:**"]
    for artigo in artigos:
        linhas.append(f"- [{artigo['titulo']}]({artigo['url']}) — score {artigo['score']}")
    return "\n".join(linhas)

def responder(pergunta, historico=None):
    pergunta = (pergunta or "").strip()
    if not pergunta:
        return "Digite uma pergunta sobre Educação Inclusiva."

    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    if not pergunta_dentro_escopo(pergunta) and not artigos:
        return "Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?"

    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nURL: {a['url']}\nTrecho: {a['trecho']}\nScore: {a['score']}"
        for a in artigos
    ) or "Nenhum trecho relevante foi encontrado na base atual."

    instrucao_usuario = (
        "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
        "Não invente fontes. Caso a informação não esteja nos trechos, diga que a base atual é insuficiente.\n\n"
        f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
        f"PERGUNTA DO USUÁRIO: {pergunta}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instrucao_usuario},
    ]

    resposta = chamar_groq(messages)
    if not resposta:
        resposta = resposta_demo_sem_llm(pergunta, artigos)

    return resposta + montar_fontes(artigos)

print("✅ Funções criadas")

✅ Funções criadas


## 9. Testes automáticos do protótipo




In [ ]:
testes = [
    ("Dentro do escopo", "O que é Atendimento Educacional Especializado (AEE)?"),
    ("Dentro do escopo", "Como adaptar atividades para alunos com autismo?"),
    ("Dentro do escopo", "Como a tecnologia assistiva ajuda estudantes com deficiência visual?"),
    ("Fora do escopo", "Qual é a capital da França?"),
    ("Caso-limite", "Como ajudar uma criança?"),
]

for categoria, pergunta in testes:
    print("=" * 80)
    print("Categoria:", categoria)
    print("Pergunta:", pergunta)
    print("-" * 80)
    print(responder(pergunta)[:1200])
    print()

Categoria: Dentro do escopo
Pergunta: O que é Atendimento Educacional Especializado (AEE)?
--------------------------------------------------------------------------------
**Resposta em modo demonstração, sem API da Groq configurada:**

Com base no artigo **O que é Atendimento Educacional Especializado (AEE)?**, a orientação principal é considerar a educação inclusiva como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado quando necessário e envolver professores, gestão, família e comunidade.

Trecho recuperado da base:
> O Atendimento Educacional Especializado, conhecido como AEE, é um serviço da educação especial que identifica, elabora e organiza recursos pedagógicos e de acessibilidade para eliminar barreiras à participação dos estudantes. Ele deve complementar ou suplementar a escolarização, nunca substituir a sala de aula comum. O

## 10. Interface de chat com Gradio


In [3]:
import gradio as gr
import base64

CAMINHO_CORDAO = "./imagens_do_tcc/CordaoTEA.svg"

with open(CAMINHO_CORDAO, "rb") as arquivo:
    cordao_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

CORDÃO_TEA_URL = f"data:image/svg+xml;base64,{cordao_base64}"

CAMINHO_ICONE_GRUPO = "./imagens_do_tcc/images.svg"

with open(CAMINHO_ICONE_GRUPO, "rb") as arquivo:
    icone_grupo_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

ICONE_GRUPO_URL = f"data:image/svg+xml;base64,{icone_grupo_base64}"

css_personalizado = f"""
body {{
    background: #FAFAFA;
}}

.gradio-container {{
    max-width: 1200px !important;
    margin: auto !important;
    font-family: 'Inter', sans-serif !important;
    background: #FAFAFA !important;
}}

/* Caixa superior */
#hero {{
    background: #FFFFFF;
    border-radius: 26px;
    padding: 34px;
    color: #111827;
    box-shadow: 0 8px 24px rgba(0, 0, 0, 0.05);
    margin-bottom: 24px;
    position: relative;
    overflow: hidden;
}}

/* Cordão como borda decorativa do topo */
#hero::before {{
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 26px;
    padding: 10px;
    background-image: url("{CORDÃO_TEA_URL}");
    background-size: 180px;
    background-repeat: repeat;
    background-position: center;
    opacity: 0.95;

    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;

    pointer-events: none;
}}

#hero h1 {{
    font-size: 32px;
    margin-bottom: 8px;
    color: #111827;
}}

#hero p {{
    font-size: 16px;
    line-height: 1.6;
    color: #374151;
}}

/* Caixa do chat */
#chatbot {{
    border-radius: 24px !important;
    background: #FFFFFF !important;
    box-shadow: 0 8px 24px rgba(0,0,0,0.05);
    position: relative;
    overflow: hidden;
}}

/* Borda do cordão no chat */
#chatbot::before {{
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 24px;
    padding: 9px;
    background-image: url("{CORDÃO_TEA_URL}");
    background-size: 160px;
    background-repeat: repeat;
    background-position: center;
    opacity: 0.95;

    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;

    pointer-events: none;
    z-index: 2;
}}

/* Cards laterais */
.card-info {{
    background: #FFFFFF;
    border-radius: 22px;
    padding: 22px;
    box-shadow: 0 6px 18px rgba(0, 0, 0, 0.05);
    position: relative;
    overflow: hidden;
}}

.card-info::before {{
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 22px;
    padding: 8px;
    background-image: url("{CORDÃO_TEA_URL}");
    background-size: 140px;
    background-repeat: repeat;
    background-position: center;
    opacity: 0.95;

    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;

    pointer-events: none;
}}

.card-info h3 {{
    color: #111827;
    margin-top: 0;
}}

.small-text {{
    font-size: 14px;
    color: #374151;
    line-height: 1.6;
}}

/* Campo de pergunta */
textarea {{
    border-radius: 14px !important;
    border: 1px solid #D1D5DB !important;
    background: #FFFFFF !important;
}}

/* Botões */
button {{
    border-radius: 14px !important;
    font-weight: 600 !important;
}}

/* Botão enviar */
#btn-enviar {{
    background: #111827 !important;
    color: white !important;
    border: none !important;
}}

#btn-enviar:hover {{
    background: #000000 !important;
}}

/* Botão limpar */
#btn-limpar {{
    background: #FFFFFF !important;
    color: #111827 !important;
    border: 1px solid #D1D5DB !important;
}}

/* Perguntas sugeridas */
.pergunta-btn button {{
    background: #FFFFFF !important;
    color: #111827 !important;
    border: 1px solid #D1D5DB !important;
    box-shadow: 0 3px 10px rgba(0,0,0,0.06);
}}

.pergunta-btn button:hover {{
    background: #F3F4F6 !important;
}}

.footer {{
    text-align: center;
    color: #6B7280;
    font-size: 13px;
    margin-top: 18px;
}}

.icone-grupo {{
    width: 44px;
    height: 44px;
    object-fit: contain;
    margin-right: 12px;
}}

.titulo-grupo {{
    display: flex;
    align-items: center;
    gap: 10px;
    margin: 0;
    color: #111827;
    font-size: 32px;
}}
"""
# ============================================================
# FUNÇÃO DO CHAT
# Pergunta
# ============================================================

def conversar(pergunta, historico):
    if historico is None:
        historico = []

    if pergunta is None or pergunta.strip() == "":
        return "", historico

    resposta = responder(pergunta)

    historico.append((pergunta, resposta))

    return "", historico


def limpar_chat():
    return []


# ============================================================
# TEMA DO GRADIO
# ============================================================

tema = gr.themes.Soft(
    primary_hue="green",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter")
)

# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    theme=tema,
    css=css_personalizado,
    title="Assistente Diversa — Educação Inclusiva"
) as demo:

    gr.HTML(f"""
<div id="hero">
    <h1 class="titulo-grupo">
        <img src="{ICONE_GRUPO_URL}" class="icone-grupo">
        <span>Equipe Hélice</span>
    </h1>

    <p>
        Protótipo funcional para TCC 
    </p>
</div>
""")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversa com o Assistente",
                elem_id="chatbot",
                height=520
            )

            pergunta = gr.Textbox(
                label="Digite sua pergunta",
                placeholder="Exemplo: O que é AEE? Como incluir um aluno com TEA?",
                lines=2,
                elem_id="pergunta-textbox" # Added elem_id for specific styling
            )

            with gr.Row():
                btn_enviar = gr.Button("Enviar pergunta", elem_id="btn-enviar")
                btn_limpar = gr.Button("Limpar conversa", elem_id="btn-limpar")

        with gr.Column(scale=1):
            gr.HTML("""
            <div class="card-info">
                <h3>📚 Sobre o protótipo</h3>
                <p class="small-text">
                    Este assistente responde com base em uma base de conhecimento
                    cadastrada manualmente com fontes reais sobre educação inclusiva.
                </p>
                <p class="small-text">
                    Ele não substitui professores, profissionais do AEE, médicos,
                    psicólogos ou especialistas.
                </p>
            </div>
            """)

            gr.HTML("<br>")

            gr.HTML("""
            <div class="card-info">
                <h3>✅ Perguntas sugeridas</h3>
                <p class="small-text">Clique em uma pergunta para preencher o campo de texto.</p>
            </div>
            """)

            btn_p1 = gr.Button("O que é educação inclusiva?", elem_classes="pergunta-btn")
            btn_p2 = gr.Button("O que é AEE?", elem_classes="pergunta-btn")
            btn_p3 = gr.Button("Como incluir aluno com TEA?", elem_classes="pergunta-btn")
            btn_p4 = gr.Button("O que diz a LBI sobre inclusão?", elem_classes="pergunta-btn")
            btn_p5 = gr.Button("O que são tecnologias assistivas?", elem_classes="pergunta-btn")

    gr.HTML("""
    <div class="footer">
        Projeto acadêmico desenvolvido para fins educacionais — SoulCode / Accenture
    </div>
    """)

    # Eventos dos botões principais
    btn_enviar.click(
        fn=conversar,
        inputs=[pergunta, chatbot],
        outputs=[pergunta, chatbot]
    )

    pergunta.submit(
        fn=conversar,
        inputs=[pergunta, chatbot],
        outputs=[pergunta, chatbot]
    )

    btn_limpar.click(
        fn=limpar_chat,
        inputs=None,
        outputs=chatbot
    )

    # Eventos das perguntas sugeridas
    btn_p1.click(lambda: "O que é educação inclusiva?", outputs=pergunta)
    btn_p2.click(lambda: "O que é Atendimento Educacional Especializado?", outputs=pergunta)
    btn_p3.click(lambda: "Como incluir um aluno com TEA em sala de aula?", outputs=pergunta)
    btn_p4.click(lambda: "O que a Lei Brasileira de Inclusão fala sobre educação?", outputs=pergunta)
    btn_p5.click(lambda: "O que são tecnologias assistivas na educação inclusiva?", outputs=pergunta)


demo.launch(share=True)

C:\Users\Cesar\AppData\Local\Temp\ipykernel_7484\4237269681.py:253: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d2f0dbd19735d2e87f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
